In [1]:
import os
import json
import re
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv


load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="REDACTED-SEE-.env-AT-REPO-ROOT",
)

In [2]:
# CLASSLA-PIQA datasets
# Place the corresponding *.tsv files in the datasets/ folder
# e.g. datasets/piqa-en.tsv, datasets/piqa-sl.tsv, etc.
# Expected TSV columns: prompt, sol1, sol2, label
tests = [
    #"ckm_latin",
    #"eng_latin",
    #"hrv_latin",
    #"mkd_cyrl",
    #"slv_latin_cerk",
    #"slv_latin",
    #"srp_cyrl",
    #"srp_latin",
    #"srp_tor_cyrl",
    "srp_tor_latin",
    #"sl_prl"
]

# OpenRouter models
models = [
    #"google/gemini-3.1-pro-preview",
    #"anthropic/claude-opus-4.6",
    #"google/gemini-3.1-flash-lite-preview",
    #"google/gemini-3-flash-preview"
    #"anthropic/claude-sonnet-4.6",
    #"openai/gpt-5",
    #"google/gemini-2.5-pro",
    #"google/gemini-2.5-flash",
    #"openai/gpt-4o",
    #"anthropic/claude-haiku-4.5",
    #"mistralai/mistral-medium-3.1",
    "meta-llama/llama-3.3-70b-instruct",
    #"google/gemma-4-31b-it",
    #"google/gemma-4-26b-a4b-it"
    #"google/gemma-3-27b-it",
    #"qwen/qwen3-32b",
    #"openai/gpt-3.5-turbo",
    #"openai/gpt-5.4-pro",
    #"openai/gpt-5.4",
    #"mistralai/mistral-large-2512",
    #"mistralai/mistral-small-2603",
    #"meta-llama/llama-4-maverick"
]

In [3]:
PURGE_FAILED = True

def is_failed_submission(path, fail_ratio=0.9):
    """Return True if a submission file is mostly sentinels (failed run)."""
    try:
        with open(path) as f:
            data = json.load(f)
        preds = data["predictions"][0]["predictions"]
        if not preds:
            return True
        n2 = sum(1 for p in preds if p == 2)
        return (n2 / len(preds)) >= fail_ratio
    except Exception:
        return True  # unreadable/corrupt -> treat as failed

def extract_answer(raw):
    """Two-stage parse: strict JSON first, then regex fallback.
    Reasoning models often wrap or precede the JSON with prose, so a bare
    json.loads is not enough. Returns 0/1, or None if nothing parseable."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()

    # Stage 1: strict JSON on the first complete {...} object
    candidate = raw
    if not candidate.startswith("{"):
        m = re.search(r'\{[^{}]*\}', candidate)
        candidate = m.group() if m else None
    if candidate:
        try:
            val = int(json.loads(candidate)["answer"])
            if val in (0, 1):
                return val
        except Exception:
            pass

    # Stage 2: regex fallback over the whole response
    m = re.search(r'"?\'?answer"?\'?\s*:\s*([01])', raw)
    if m:
        return int(m.group(1))

    return None

def predict_gpt(df_test_name, gpt_model):
    os.makedirs("piqa_submissions", exist_ok=True)
    model_name = gpt_model.split("/")[1]
    out_path = f"piqa_submissions/submission-{model_name}-{df_test_name}.json"

    if os.path.exists(out_path):
        if PURGE_FAILED and is_failed_submission(out_path):
            print(f"Purging failed submission, will regenerate: {out_path}")
            os.remove(out_path)
        else:
            print(f"Skipping (already exists): {out_path}")
            return

    tsv_path = f"piqa_datasets/{df_test_name}.tsv"
    if not os.path.exists(tsv_path):
        print(f"WARNING: Missing {tsv_path}, skipping.")
        return
    df = pd.read_csv(tsv_path, sep="\t")

    responses = []
    n_api_fail = 0
    n_parse_fail = 0
    start_time = time.time()

    for _, entry in df.iterrows():
        prompt = (
            f"### Task\n"
            f"    Given the following situation, which option is more likely to be correct?\n\n"
            f"    Situation: {entry['prompt']}\n\n"
            f"    Option 0: {entry['solution0']}\n\n"
            f"    Option 1: {entry['solution1']}\n\n"
            f"### Output format\n"
            f"    Return a valid JSON dictionary with the following key: 'answer' "
            f"and a value should be either 0 (if option 0 is more plausible) "
            f"or 1 (if option 1 is more plausible). "
            f"Answer ONLY with the JSON dictionary, no explanation."
        )

        completion = None
        for attempt in range(4):
            try:
                completion = client.chat.completions.create(
                    model=gpt_model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0,
                    max_tokens=4096,  # headroom for reasoning models; the visible JSON is tiny but thinking tokens count against this
                    extra_body={
                                "provider": {
                                    "allow_fallbacks": True,
                                    "order": ["deepinfra", "parasail", "novita"],
                                    "data_collection": "allow",
                                }
                            },
                )
                break
            except Exception as e:
                if "429" in str(e) and attempt < 3:
                    wait = [30, 60, 90][attempt]
                    print(f"Rate limited, waiting {wait}s... (attempt {attempt+1}/4)")
                    time.sleep(wait)
                else:
                    print(f"API error, recording sentinel: {e}")
                    break

        if completion is None:
            responses.append(2)
            n_api_fail += 1
            continue

        try:
            content = completion.choices[0].message.content
            if content is None or not content.strip():
                # empty content usually means the output was truncated before any
                # visible tokens (finish_reason='length') — flag it loudly
                fr = completion.choices[0].finish_reason
                print(f"Empty content (finish_reason={fr}); recording sentinel")
                responses.append(2)
                n_parse_fail += 1
                continue
            predicted = extract_answer(content)
            if predicted is None:
                print(f"Error extracting label from: {content[:80]!r}")
                responses.append(2)
                n_parse_fail += 1
            else:
                responses.append(predicted)
        except Exception as e:
            print(f"Error extracting label: {e}")
            responses.append(2)
            n_parse_fail += 1

    elapsed = time.time() - start_time
    n = len(responses)
    if n > 0:
        print(f"Done. {elapsed/60:.2f} min | {elapsed/n:.3f} s/instance "
              f"| api_fail={n_api_fail} parse_fail={n_parse_fail} "
              f"({(n_api_fail+n_parse_fail)/n:.1%} sentinel)")
    else:
        print(f"Done. {elapsed/60:.2f} min | 0 instances")

    with open(out_path, "w") as f:
        json.dump({"system": gpt_model,
                   "predictions": [{"train": "NA (zero-shot)",
                                    "test": df_test_name,
                                    "predictions": responses}]}, f)
    print(f"Saved: {out_path}")

# Run all models on all datasets
# Tip: comment out models or datasets you want to skip / re-run individually
for model in models:
    for test in tests:
        print(f"\n=== {model} | {test} ===")
        predict_gpt(test, model)


=== meta-llama/llama-3.3-70b-instruct | srp_tor_latin ===
Done. 1.25 min | 0.751 s/instance | api_fail=0 parse_fail=0 (0.0% sentinel)
Saved: piqa_submissions/submission-llama-3.3-70b-instruct-srp_tor_latin.json
